# Практика: Python

Укажите **ФИО**, **группу** и **кредит** → нажмите **Сгенерировать задание** → напишите решение в ячейке ниже → нажмите **Проверить**. Результат: оценка и комментарий (ошибки при наличии).

---
**Настройка (только для преподавателя).** Запустите следующую ячейку **один раз** перед началом работы. Ключ API подгружается с backend (exam_core.py с GitHub). Студентам переходить к ячейке с полями **ФИО, Группа, Кредит**.

In [ ]:
!pip install -q openai ipywidgets

import urllib.request
import hashlib
import ipywidgets as widgets
from IPython.display import display, clear_output

# Код и ключ API с GitHub (exam_core.py) и backend
url = "https://raw.githubusercontent.com/argokz/exam-kmu/main/exam_core.py"
exec(urllib.request.urlopen(url).read().decode(), globals())

current_assignment = ''
current_variant = 1
current_fio = ''
current_group = ''
current_credit = ''
print('Готово.')

---
## ФИО, группа, кредит

In [ ]:
fio_in = widgets.Text(placeholder="ФИО", description="ФИО:", style={"description_width": "60px"})
group_in = widgets.Text(placeholder="Группа", description="Группа:", style={"description_width": "60px"})
credit_in = widgets.Text(placeholder="Кредит", description="Кредит:", style={"description_width": "60px"})
out_assign = widgets.Output()

def on_generate(b):
    global current_assignment, current_variant, current_fio, current_group, current_credit
    fio, group, credit = fio_in.value.strip(), group_in.value.strip(), credit_in.value.strip()
    if not fio or not group or not credit:
        with out_assign:
            clear_output(wait=True)
            print("Заполните ФИО, группу и кредит.")
        return
    h = int(hashlib.sha256(f"{credit}{group}{fio}".encode()).hexdigest(), 16) % 17
    current_variant = h + 1
    current_fio, current_group, current_credit = fio, group, credit
    with out_assign:
        clear_output(wait=True)
        print("Генерация задания...")
    current_assignment = generate_assignment(current_variant)
    with out_assign:
        clear_output(wait=True)
        print(f"Вариант {current_variant}")
        print("-" * 40)
        print(current_assignment)

btn_gen = widgets.Button(description="Сгенерировать задание", button_style="primary")
btn_gen.on_click(on_generate)
display(widgets.VBox([fio_in, group_in, credit_in, btn_gen, out_assign]))

Напишите решение в ячейке ниже (переменная `student_code`) и **запустите ячейку**, затем нажмите **Проверить**.

In [ ]:
student_code = """
# Ваше решение
pass
"""

## Проверить задание

In [ ]:
out_check = widgets.Output()

def on_check(b):
    global current_assignment, current_variant, current_fio, current_group, current_credit
    with out_check:
        clear_output(wait=True)
        print("Проверка...")
    code = ""
    try:
        code = get_ipython().user_ns.get("student_code", "")
    except Exception:
        code = ""
    if not code or code.strip() in ("", "pass", "# Ваше решение"):
        with out_check:
            clear_output(wait=True)
            print("Сначала напишите решение в ячейке выше и запустите её.")
        return
    if not current_assignment:
        with out_check:
            clear_output(wait=True)
            print("Сначала нажмите «Сгенерировать задание».")
        return
    result = evaluate_solution(current_assignment, code, current_variant)
    try:
        save_to_sqlite(current_fio, current_group, current_credit, current_variant, current_assignment, code, result["score"], result["feedback"])
    except Exception:
        pass
    with out_check:
        clear_output(wait=True)
        print(f"Оценка: {result['score']} / 100")
        print(f"Комментарий (ошибки при наличии): {result['feedback']}")

btn_check = widgets.Button(description="Проверить", button_style="success")
btn_check.on_click(on_check)
display(widgets.VBox([btn_check, out_check]))